# WESAD Mental State Classification: SNN vs CNN
**Mayuri Thorat — Örebro PhD Application**

This notebook classifies mental states (baseline, stress, amusement) from EDA, ECG, ACC signals using Spiking Neural Networks vs CNN baseline on the WESAD dataset.

**Before running:** Upload the WESAD dataset to your Google Drive and update `WESAD_PATH` in Cell 3.

In [1]:
# CELL 1 — Install dependencies
!pip install snntorch -q
print('Done')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.6/125.6 kB 4.4 MB/s eta 0:00:00
Done


In [2]:
# CELL 2 — Imports
import numpy as np
import pickle, os, warnings
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import LeaveOneGroupOut
from scipy.signal import butter, filtfilt
from scipy.stats import skew, kurtosis
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
print('All imports OK')
print('GPU available:', torch.cuda.is_available())

All imports OK
GPU available: True


In [4]:
# CELL 3 — Download WESAD from Kaggle
!pip install kagglehub -q

import kagglehub
path = kagglehub.dataset_download("orvile/wesad-wearable-stress-affect-detection-dataset")
WESAD_PATH = path
print("WESAD path:", WESAD_PATH)

Using Colab cache for faster access to the 'wesad-wearable-stress-affect-detection-dataset' dataset.
WESAD path: /kaggle/input/wesad-wearable-stress-affect-detection-dataset


In [5]:
# CELL 4 — Feature extraction functions
def extract_features(signal, fs=700):
    feats = []
    feats.append(np.mean(signal))
    feats.append(np.std(signal))
    feats.append(np.min(signal))
    feats.append(np.max(signal))
    feats.append(np.sqrt(np.mean(signal**2)))
    feats.append(skew(signal))
    feats.append(kurtosis(signal))
    feats.append(np.percentile(signal, 25))
    feats.append(np.percentile(signal, 75))
    feats.append(np.percentile(signal, 75) - np.percentile(signal, 25))
    fft_vals = np.abs(np.fft.rfft(signal))
    freqs = np.fft.rfftfreq(len(signal), 1/fs)
    feats.append(np.sum(fft_vals))
    for low, high in [(0,0.5),(0.5,2),(2,8),(8,16)]:
        mask = (freqs>=low) & (freqs<high)
        feats.append(np.sum(fft_vals[mask]) if mask.any() else 0)
    return np.array(feats)

def extract_eda_features(eda, fs=700):
    b, a = butter(2, 0.05/(fs/2), btype='low')
    scl = filtfilt(b, a, eda)
    scr = eda - scl
    base = extract_features(eda, fs)
    tonic = [np.mean(scl), np.std(scl), np.max(scl)]
    phasic = [np.mean(np.abs(scr)), np.std(scr), float(np.sum(scr > np.std(scr)))]
    return np.concatenate([base, tonic, phasic])

def extract_ecg_features(ecg, fs=700):
    base = extract_features(ecg, fs)
    threshold = np.mean(ecg) + 1.5*np.std(ecg)
    peaks = []
    min_dist = int(0.3 * fs)
    i = 0
    while i < len(ecg):
        if ecg[i] > threshold:
            region = ecg[i:i+min_dist]
            if len(region) > 0:
                peaks.append(i + np.argmax(region))
                i += min_dist
            else:
                i += 1
        else:
            i += 1
    if len(peaks) > 1:
        rr = np.diff(peaks) / fs * 1000
        hr = 60000 / np.mean(rr) if np.mean(rr) > 0 else 0
        rmssd = np.sqrt(np.mean(np.diff(rr)**2)) if len(rr)>1 else 0
        sdnn = np.std(rr) if len(rr)>1 else 0
    else:
        hr, rmssd, sdnn = 0, 0, 0
    return np.concatenate([base, [hr, rmssd, sdnn]])

def extract_acc_features(ax, ay, az, fs=700):
    mag = np.sqrt(ax**2 + ay**2 + az**2)
    return np.concatenate([extract_features(ax,fs), extract_features(ay,fs),
                           extract_features(az,fs), extract_features(mag,fs)])

print('Feature extraction functions ready')

Feature extraction functions ready


In [ ]:
# CELL 5 — Load all WESAD subjects
def load_all_subjects(wesad_path, window_sec=60, step_sec=30):
    subject_ids = [2,3,4,5,6,7,8,9,10,11,13,14,15,16,17]
    label_map = {1:0, 2:1, 3:2}
    all_features, all_labels, all_subjects = [], [], []

    for sid in subject_ids:
        try:
            pkl_path = os.path.join(wesad_path, f'S{sid}', f'S{sid}.pkl')
            with open(pkl_path, 'rb') as f:
                data = pickle.load(f, encoding='latin1')
            chest = data['signal']['chest']
            eda = chest['EDA'].flatten()
            ecg = chest['ECG'].flatten()
            acc_x = chest['ACC'][:,0]
            acc_y = chest['ACC'][:,1]
            acc_z = chest['ACC'][:,2]
            labels_raw = data['label'].flatten()
            valid_mask = np.isin(labels_raw, [1,2,3])
            fs = 700
            w = int(window_sec * fs)
            s = int(step_sec * fs)

            for start in range(0, len(eda) - w, s):
                end = start + w
                seg_labels = labels_raw[start:end]
                if not np.all(valid_mask[start:end]):
                    continue
                unique, counts = np.unique(seg_labels, return_counts=True)
                majority = unique[np.argmax(counts)]
                if majority not in [1,2,3]:
                    continue
                if np.max(counts)/len(seg_labels) < 0.8:
                    continue
                f_eda = extract_eda_features(eda[start:end], fs)
                f_ecg = extract_ecg_features(ecg[start:end], fs)
                f_acc = extract_acc_features(acc_x[start:end], acc_y[start:end], acc_z[start:end], fs)
                feat = np.concatenate([f_eda, f_ecg, f_acc])
                if np.any(np.isnan(feat)) or np.any(np.isinf(feat)):
                    continue
                all_features.append(feat)
                all_labels.append(label_map[majority])
                all_subjects.append(sid)

            print(f'S{sid}: OK')
        except Exception as e:
            print(f'S{sid}: ERROR - {e}')

    X = np.array(all_features)
    y = np.array(all_labels)
    groups = np.array(all_subjects)
    print(f'\nTotal windows: {len(X)}, Features per window: {X.shape[1]}')
    print(f'Class counts — 0=Baseline: {np.sum(y==0)}, 1=Stress: {np.sum(y==1)}, 2=Amusement: {np.sum(y==2)}')
    return X, y, groups

X, y, groups = load_all_subjects(WESAD_PATH)
print('Data loaded successfully')

In [6]:
# CELL 6 — SNN model definition
class SNN(nn.Module):
    def __init__(self, input_size, hidden_size=256, output_size=3, num_steps=50, beta=0.95):
        super().__init__()
        self.num_steps = num_steps
        spike_grad = surrogate.fast_sigmoid(slope=25)
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        self.fc2 = nn.Linear(hidden_size, hidden_size//2)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        self.fc3 = nn.Linear(hidden_size//2, output_size)
        self.lif3 = snn.Leaky(beta=beta, spike_grad=spike_grad)

    def forward(self, x):
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        spk3_rec, mem3_rec = [], []
        for _ in range(self.num_steps):
            spk1, mem1 = self.lif1(self.fc1(x), mem1)
            spk2, mem2 = self.lif2(self.fc2(spk1), mem2)
            spk3, mem3 = self.lif3(self.fc3(spk2), mem3)
            spk3_rec.append(spk3)
            mem3_rec.append(mem3)
        return torch.stack(spk3_rec), torch.stack(mem3_rec)

print('SNN defined')

SNN defined


In [7]:
# CELL 7 — CNN baseline definition
class CNN(nn.Module):
    def __init__(self, input_size, output_size=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x):
        return self.net(x)

print('CNN defined')

CNN defined


In [8]:
# CELL 8 — Training and prediction functions
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

def train_snn(model, Xtr, ytr, epochs=50, lr=1e-3, bs=32):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    Xt = torch.FloatTensor(Xtr).to(device)
    yt = torch.LongTensor(ytr).to(device)
    model.train()
    for ep in range(epochs):
        perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), bs):
            idx = perm[i:i+bs]
            xb, yb = Xt[idx], yt[idx]
            opt.zero_grad()
            spk_rec, _ = model(xb)
            loss_fn(spk_rec.sum(0), yb).backward()
            opt.step()
    return model

def predict_snn(model, Xte):
    model.eval()
    with torch.no_grad():
        spk_rec, _ = model(torch.FloatTensor(Xte).to(device))
        return spk_rec.sum(0).argmax(1).cpu().numpy()

def train_cnn(model, Xtr, ytr, epochs=50, lr=1e-3, bs=32):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    Xt = torch.FloatTensor(Xtr).to(device)
    yt = torch.LongTensor(ytr).to(device)
    model.train()
    for ep in range(epochs):
        perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), bs):
            idx = perm[i:i+bs]
            xb, yb = Xt[idx], yt[idx]
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()
    return model

def predict_cnn(model, Xte):
    model.eval()
    with torch.no_grad():
        return model(torch.FloatTensor(Xte).to(device)).argmax(1).cpu().numpy()

print('Training functions ready')

Using device: cuda
Training functions ready


In [11]:
# CELL 9 — Run LOSO cross-validation (this takes ~1-2 hours on Colab CPU, ~30min on GPU)
logo = LeaveOneGroupOut()
input_size = X.shape[1]
snn_f1s, cnn_f1s = [], []
snn_preds_all, cnn_preds_all, y_all = [], [], []

for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups), 1):
    subj = groups[test_idx[0]]
    print(f'Fold {fold}/15 — Subject S{subj}', end=' ... ')

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    snn_model = train_snn(SNN(input_size), X_train, y_train)
    snn_pred = predict_snn(snn_model, X_test)
    snn_f1 = f1_score(y_test, snn_pred, average='macro', zero_division=0)
    snn_f1s.append(snn_f1)
    snn_preds_all.extend(snn_pred)

    cnn_model = train_cnn(CNN(input_size), X_train, y_train)
    cnn_pred = predict_cnn(cnn_model, X_test)
    cnn_f1 = f1_score(y_test, cnn_pred, average='macro', zero_division=0)
    cnn_f1s.append(cnn_f1)
    cnn_preds_all.extend(cnn_pred)

    y_all.extend(y_test)
    print(f'SNN={snn_f1:.3f}  CNN={cnn_f1:.3f}')

y_all = np.array(y_all)
snn_preds_all = np.array(snn_preds_all)
cnn_preds_all = np.array(cnn_preds_all)
print('\nDone!')

NameError: name 'X' is not defined

In [ ]:
# CELL 10 — Print all results
print('='*60)
print('FINAL RESULTS')
print('='*60)
print(f'\nSNN  Mean macro F1: {np.mean(snn_f1s)*100:.1f}% ± {np.std(snn_f1s)*100:.1f}%')
print(f'CNN  Mean macro F1: {np.mean(cnn_f1s)*100:.1f}% ± {np.std(cnn_f1s)*100:.1f}%')
print('\n--- SNN Classification Report ---')
print(classification_report(y_all, snn_preds_all, target_names=['Baseline','Stress','Amusement']))
print('\n--- CNN Classification Report ---')
print(classification_report(y_all, cnn_preds_all, target_names=['Baseline','Stress','Amusement']))

In [ ]:
# CELL 11 — Confusion matrix plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
label_names = ['Baseline', 'Stress', 'Amusement']

for ax, preds, title in zip(axes,
    [snn_preds_all, cnn_preds_all],
    ['SNN — Ours', 'CNN Baseline']):
    cm = confusion_matrix(y_all, preds, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=label_names, yticklabels=label_names, ax=ax)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.suptitle('Confusion Matrices: Mental State Classification (WESAD LOSO)', fontsize=13)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrices.png')

In [ ]:
# CELL 12 — Per-fold F1 bar chart
fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(1, 16)
w = 0.35
ax.bar(x - w/2, snn_f1s, w, label='SNN (Ours)', color='#1D9E75', alpha=0.85)
ax.bar(x + w/2, cnn_f1s, w, label='CNN Baseline', color='#378ADD', alpha=0.85)
ax.axhline(np.mean(snn_f1s), color='#1D9E75', linestyle='--', linewidth=1.5,
           label=f'SNN Mean: {np.mean(snn_f1s):.3f}')
ax.axhline(np.mean(cnn_f1s), color='#378ADD', linestyle='--', linewidth=1.5,
           label=f'CNN Mean: {np.mean(cnn_f1s):.3f}')
ax.set_xlabel('Fold (Leave-One-Subject-Out)', fontsize=12)
ax.set_ylabel('Macro F1 Score', fontsize=12)
ax.set_title('Per-fold Macro F1: SNN vs CNN on WESAD', fontsize=13)
ax.set_xticks(x)
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('f1_per_fold.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: f1_per_fold.png')

In [ ]:
# CELL 13 — COPY THESE NUMBERS FOR YOUR PAPER
print('='*60)
print('COPY THESE NUMBERS — paste back to Claude')
print('='*60)
print(f'SNN macro F1: {np.mean(snn_f1s)*100:.1f}% +/- {np.std(snn_f1s)*100:.1f}%')
print(f'CNN macro F1: {np.mean(cnn_f1s)*100:.1f}% +/- {np.std(cnn_f1s)*100:.1f}%')
r = classification_report(y_all, snn_preds_all,
    target_names=['Baseline','Stress','Amusement'], output_dict=True)
print(f"SNN Baseline F1: {r['Baseline']['f1-score']*100:.1f}%")
print(f"SNN Stress F1:   {r['Stress']['f1-score']*100:.1f}%")
print(f"SNN Amusement F1:{r['Amusement']['f1-score']*100:.1f}%")
r2 = classification_report(y_all, cnn_preds_all,
    target_names=['Baseline','Stress','Amusement'], output_dict=True)
print(f"CNN Baseline F1: {r2['Baseline']['f1-score']*100:.1f}%")
print(f"CNN Stress F1:   {r2['Stress']['f1-score']*100:.1f}%")
print(f"CNN Amusement F1:{r2['Amusement']['f1-score']*100:.1f}%")
print('\nAlso download and send me:')
print('  - confusion_matrices.png')
print('  - f1_per_fold.png')